In [1]:
# https://www.kaggle.com/datasets/smid80/weatherww2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [3]:
df = pd.read_csv("00_files/03_weather_ww2_merged.csv", low_memory=False)

In [4]:
df.head()

,STA,Date,Precip,MaxTemp,MinTemp,MeanTemp,Snowfall,YR,MO,DA,PRCP,MAX,MIN,MEA,SNF,NAME,STATE/COUNTRY ID,ELEV,Latitude,Longitude
0,10001,1942-7-1,1.016,25.555556,22.222222,23.888889,0,42,7,1,0.04,78.0,72.0,75.0,0,ACCRA,GH,62,5.6,-0.3
1,10001,1942-7-2,0,28.888889,21.666667,25.555556,0,42,7,2,0,84.0,71.0,78.0,0,ACCRA,GH,62,5.6,-0.3
2,10001,1942-7-3,2.54,26.111111,22.222222,24.444444,0,42,7,3,0.1,79.0,72.0,76.0,0,ACCRA,GH,62,5.6,-0.3
3,10001,1942-7-4,2.54,26.666667,22.222222,24.444444,0,42,7,4,0.1,80.0,72.0,76.0,0,ACCRA,GH,62,5.6,-0.3
4,10001,1942-7-5,0,26.666667,21.666667,24.444444,0,42,7,5,0,80.0,71.0,76.0,0,ACCRA,GH,62,5.6,-0.3


In [5]:
df["Date"] = pd.to_datetime(df["Date"])

In [6]:
df.isna().sum()

STA                    0
Date                   0
Precip                 0
MaxTemp                0
MinTemp                0
MeanTemp               0
Snowfall            1163
YR                     0
MO                     0
DA                     0
PRCP                1932
MAX                  474
MIN                  468
MEA                  498
SNF                 1163
NAME                   0
STATE/COUNTRY ID       0
ELEV                   0
Latitude               0
Longitude              0
dtype: int64

In [7]:
df = df.dropna(subset=["MAX", "MIN", "MEA", "Snowfall", "SNF", "PRCP"])

In [8]:
df.isna().sum()

STA                 0
Date                0
Precip              0
MaxTemp             0
MinTemp             0
MeanTemp            0
Snowfall            0
YR                  0
MO                  0
DA                  0
PRCP                0
MAX                 0
MIN                 0
MEA                 0
SNF                 0
NAME                0
STATE/COUNTRY ID    0
ELEV                0
Latitude            0
Longitude           0
dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 115697 entries, 0 to 119039
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   STA               115697 non-null  int64         
 1   Date              115697 non-null  datetime64[ns]
 2   Precip            115697 non-null  object        
 3   MaxTemp           115697 non-null  float64       
 4   MinTemp           115697 non-null  float64       
 5   MeanTemp          115697 non-null  float64       
 6   Snowfall          115697 non-null  object        
 7   YR                115697 non-null  int64         
 8   MO                115697 non-null  int64         
 9   DA                115697 non-null  int64         
 10  PRCP              115697 non-null  object        
 11  MAX               115697 non-null  float64       
 12  MIN               115697 non-null  float64       
 13  MEA               115697 non-null  float64       
 14  SNF      

In [10]:
cols_to_float = ["Precip", "Snowfall", "PRCP", "SNF"]

for c in cols_to_float:
    df[c] = df[c].replace("T", 0)
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(0)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 115697 entries, 0 to 119039
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   STA               115697 non-null  int64         
 1   Date              115697 non-null  datetime64[ns]
 2   Precip            115697 non-null  float64       
 3   MaxTemp           115697 non-null  float64       
 4   MinTemp           115697 non-null  float64       
 5   MeanTemp          115697 non-null  float64       
 6   Snowfall          115697 non-null  float64       
 7   YR                115697 non-null  int64         
 8   MO                115697 non-null  int64         
 9   DA                115697 non-null  int64         
 10  PRCP              115697 non-null  float64       
 11  MAX               115697 non-null  float64       
 12  MIN               115697 non-null  float64       
 13  MEA               115697 non-null  float64       
 14  SNF      

In [12]:
cols_to_drop = ["YR", "MO", "DA", "MAX", "MIN", "MEA"]
df = df.drop(columns=cols_to_drop)
df.head()

,STA,Date,Precip,MaxTemp,MinTemp,MeanTemp,Snowfall,PRCP,SNF,NAME,STATE/COUNTRY ID,ELEV,Latitude,Longitude
0,10001,1942-07-01,1.016,25.555556,22.222222,23.888889,0.0,0.04,0.0,ACCRA,GH,62,5.6,-0.3
1,10001,1942-07-02,0.000,28.888889,21.666667,25.555556,0.0,0.00,0.0,ACCRA,GH,62,5.6,-0.3
2,10001,1942-07-03,2.540,26.111111,22.222222,24.444444,0.0,0.10,0.0,ACCRA,GH,62,5.6,-0.3
3,10001,1942-07-04,2.540,26.666667,22.222222,24.444444,0.0,0.10,0.0,ACCRA,GH,62,5.6,-0.3
4,10001,1942-07-05,0.000,26.666667,21.666667,24.444444,0.0,0.00,0.0,ACCRA,GH,62,5.6,-0.3


In [13]:
df = df.sort_values(["STA", "Date"])

df["MeanTemp_lag1"] = df.groupby("STA")["MeanTemp"].shift(1)
df["MeanTemp_lag2"] = df.groupby("STA")["MeanTemp"].shift(2)

df = df.dropna(subset=["MeanTemp_lag1", "MeanTemp_lag2"])

df[["STA", "Date", "MeanTemp", "MeanTemp_lag1", "MeanTemp_lag2"]].head()

,STA,Date,MeanTemp,MeanTemp_lag1,MeanTemp_lag2
2,10001,1942-07-03,24.444444,25.555556,23.888889
3,10001,1942-07-04,24.444444,24.444444,25.555556
4,10001,1942-07-05,24.444444,24.444444,24.444444
5,10001,1942-07-06,24.444444,24.444444,24.444444
6,10001,1942-07-07,25.555556,24.444444,24.444444


In [14]:
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["dayofyear"] = df["Date"].dt.dayofyear

df[["Date", "year", "month", "dayofyear"]].head()

,Date,year,month,dayofyear
2,1942-07-03,1942,7,184
3,1942-07-04,1942,7,185
4,1942-07-05,1942,7,186
5,1942-07-06,1942,7,187
6,1942-07-07,1942,7,188


In [15]:
target_col = "MeanTemp"

numeric_features = [
    "MaxTemp", "MinTemp",      
    "Precip", "Snowfall", "PRCP", "SNF", 
    "ELEV", "Latitude", "Longitude",
    "year", "month", "dayofyear",
    "MeanTemp_lag1", "MeanTemp_lag2",
]

categorical_features = [
    "STA",  
    "STATE/COUNTRY ID",
    "NAME",
]

all_features = numeric_features + categorical_features

df_model = df[all_features + [target_col]].copy()
df_model.head()

,MaxTemp,MinTemp,Precip,Snowfall,PRCP,SNF,ELEV,Latitude,Longitude,year,month,dayofyear,MeanTemp_lag1,MeanTemp_lag2,STA,STATE/COUNTRY ID,NAME,MeanTemp
2,26.111111,22.222222,2.54,0.0,0.1,0.0,62,5.6,-0.3,1942,7,184,25.555556,23.888889,10001,GH,ACCRA,24.444444
3,26.666667,22.222222,2.54,0.0,0.1,0.0,62,5.6,-0.3,1942,7,185,24.444444,25.555556,10001,GH,ACCRA,24.444444
4,26.666667,21.666667,0.00,0.0,0.0,0.0,62,5.6,-0.3,1942,7,186,24.444444,24.444444,10001,GH,ACCRA,24.444444
5,26.666667,21.666667,0.00,0.0,0.0,0.0,62,5.6,-0.3,1942,7,187,24.444444,24.444444,10001,GH,ACCRA,24.444444
6,28.333333,22.777778,0.00,0.0,0.0,0.0,62,5.6,-0.3,1942,7,188,24.444444,24.444444,10001,GH,ACCRA,25.555556


In [16]:
df_model.to_csv("00_files/04_weather_ww2_ml_ready.csv", index=False)